In [ ]:
# Install Pytorch & other libraries
%pip install torch tensorboard

# Install Transformers
%pip install transformers

# Install Hugging Face libraries
%pip install datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece

# COMMENT IN: if you are running on a GPU that supports BF16 data type and flash attn, such as NVIDIA L4 or NVIDIA A100
#%pip install flash-attn

In [4]:
from huggingface_hub import login
from google.colab import userdata

# 从 Colab Secrets 里读取你之前保存的 HF_TOKEN
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

print("登录成功 ✅")

登录成功 ✅


In [5]:
from datasets import load_dataset

# 加载数据集
dataset = load_dataset("philschmid/gretel-synthetic-text-to-sql", split="train")

# 看看数据集长什么样
print(f"数据集大小: {len(dataset)} 条样本")
print(f"\n第一条样本的字段: {list(dataset[0].keys())}")
print(f"\n第一条样本完整内容:")
print(dataset[0])

README.md:   0%|          | 0.00/737 [00:00<?, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

数据集大小: 100000 条样本

第一条样本的字段: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation']

第一条样本完整内容:
{'id': 5097, 'domain': 'forestry', 'domain_description': 'Comprehensive data on sustainable forest management, timber production, wildlife habitat, and carbon sequestration in forestry.', 'sql_complexity': 'single join', 'sql_complexity_description': 'only one join (specify inner, outer, cross)', 'sql_task_type': 'analytics and reporting', 'sql_task_type_description': 'generating reports, dashboards, and analytical insights', 'sql_prompt': 'What is the total volume of timber sold by each salesperson, sorted by salesperson?', 'sql_context': "CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id I

In [6]:
# 看看不同领域、不同复杂度的样本
for i in [0, 100, 1000, 5000]:
    sample = dataset[i]
    print(f"--- 样本 {i} ---")
    print(f"领域: {sample['domain']}")
    print(f"复杂度: {sample['sql_complexity']}")
    print(f"问题: {sample['sql_prompt']}")
    print(f"SQL: {sample['sql']}")
    print()

--- 样本 0 ---
领域: forestry
复杂度: single join
问题: What is the total volume of timber sold by each salesperson, sorted by salesperson?
SQL: SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;

--- 样本 100 ---
领域: sustainable infrastructure
复杂度: subqueries
问题: What is the maximum number of green buildings in each state in the US, constructed before 2015?
SQL: SELECT state, MAX(cnt) FROM (SELECT state, COUNT(*) AS cnt FROM green_buildings_us WHERE construction_year < 2015 GROUP BY state) AS subquery;

--- 样本 1000 ---
领域: entertainment industry
复杂度: basic SQL
问题: delete records with viewer 'Alex' in the viewership table
SQL: DELETE FROM viewership WHERE viewer = 'Alex';

--- 样本 5000 ---
领域: government policy
复杂度: single join
问题: What is the minimum age of artists who have exhibited in galleries located in the Warehouse District?
SQL: SELE

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 模型 ID
model_id = "google/gemma-3-1b-it"

# 配置 4bit 量化(QLoRA 的"Q",Quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 加载模型(4bit 量化版)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("✅ Gemma-3-1B 加载完成")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

✅ Gemma-3-1B 加载完成
模型参数量: 0.65B


In [10]:
# 从数据集里挑一条样本来测试
test_sample = dataset[0]

# 构造 prompt
system_message = "You are a text-to-SQL translator. Given a database schema and a user question, generate the correct SQL query."

user_prompt = f"""Database schema:
{test_sample['sql_context']}

Question: {test_sample['sql_prompt']}

Generate the SQL query:"""

messages = [
    {"role": "user", "content": system_message + "\n\n" + user_prompt}
]

# 把对话转换成模型能理解的 token(返回 dict 格式)
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

# 让模型生成回答
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
    )

# 解码生成的部分
input_length = inputs['input_ids'].shape[1]
response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

# 输出对比
print("=" * 60)
print("📝 用户问题:")
print(test_sample['sql_prompt'])
print()
print("🎯 标准答案 (ground truth):")
print(test_sample['sql'])
print()
print("🤖 Gemma 原始模型的回答:")
print(response)
print("=" * 60)

📝 用户问题:
What is the total volume of timber sold by each salesperson, sorted by salesperson?

🎯 标准答案 (ground truth):
SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;

🤖 Gemma 原始模型的回答:
```sql
SELECT salesperson_id, SUM(volume) AS total_volume
FROM timber_sales
GROUP BY salesperson_id
ORDER BY salesperson_id;
```


In [11]:
# 测试多条样本,看 baseline 整体表现
test_indices = [0, 100, 1000, 5000]

for idx in test_indices:
    test_sample = dataset[idx]

    user_prompt = f"""Database schema:
{test_sample['sql_context']}

Question: {test_sample['sql_prompt']}

Generate the SQL query:"""

    messages = [
        {"role": "user", "content": system_message + "\n\n" + user_prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    input_length = inputs['input_ids'].shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    print(f"\n{'='*60}")
    print(f"🔢 样本 {idx}  |  领域: {test_sample['domain']}  |  复杂度: {test_sample['sql_complexity']}")
    print(f"{'='*60}")
    print(f"📝 问题: {test_sample['sql_prompt']}")
    print(f"\n🎯 标准答案:\n{test_sample['sql']}")
    print(f"\n🤖 Gemma 回答:\n{response}")


🔢 样本 0  |  领域: forestry  |  复杂度: single join
📝 问题: What is the total volume of timber sold by each salesperson, sorted by salesperson?

🎯 标准答案:
SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;

🤖 Gemma 回答:
```sql
SELECT salesperson_id, SUM(volume) AS total_volume
FROM timber_sales
GROUP BY salesperson_id
ORDER BY salesperson_id;
```

🔢 样本 100  |  领域: sustainable infrastructure  |  复杂度: subqueries
📝 问题: What is the maximum number of green buildings in each state in the US, constructed before 2015?

🎯 标准答案:
SELECT state, MAX(cnt) FROM (SELECT state, COUNT(*) AS cnt FROM green_buildings_us WHERE construction_year < 2015 GROUP BY state) AS subquery;

🤖 Gemma 回答:
```sql
SELECT state, COUNT(*)
FROM green_buildings_us
WHERE construction_year < 2015
GROUP BY state
ORDER BY state;
```

🔢 样本 1000  |  领域: entertainment industry  |  复杂度: 

In [12]:
# Step 1: 抽样 + 切分训练/验证集
# 设置随机种子,保证每次抽样结果一样(可复现)
dataset_shuffled = dataset.shuffle(seed=42)

# 抽 5000 条训练 + 200 条验证
train_dataset = dataset_shuffled.select(range(5000))
eval_dataset = dataset_shuffled.select(range(5000, 5200))

print(f"训练集: {len(train_dataset)} 条")
print(f"验证集: {len(eval_dataset)} 条")


# Step 2: 定义格式化函数——把每条数据转成对话格式
def format_for_training(example):
    """
    把 (sql_context, sql_prompt, sql) 转成 messages 列表。
    SFTTrainer 期望的输入格式就是这种 messages 列表。
    """
    system_message = "You are a text-to-SQL translator. Given a database schema and a user question, generate the correct SQL query."

    user_content = f"""Database schema:
{example['sql_context']}

Question: {example['sql_prompt']}

Generate the SQL query:"""

    assistant_content = example['sql']

    return {
        "messages": [
            {"role": "user", "content": system_message + "\n\n" + user_content},
            {"role": "assistant", "content": assistant_content}
        ]
    }


# Step 3: 应用到整个数据集
train_dataset = train_dataset.map(format_for_training, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(format_for_training, remove_columns=eval_dataset.column_names)

# Step 4: 看看格式化后的样本
print("\n" + "="*60)
print("📋 格式化后的训练样本示例:")
print("="*60)
import json
print(json.dumps(train_dataset[0], indent=2, ensure_ascii=False))

训练集: 5000 条
验证集: 200 条


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]


📋 格式化后的训练样本示例:
{
  "messages": [
    {
      "role": "user",
      "content": "You are a text-to-SQL translator. Given a database schema and a user question, generate the correct SQL query.\n\nDatabase schema:\nCREATE TABLE Members (MemberID INT, Age INT, Gender VARCHAR(10), MembershipType VARCHAR(20)); INSERT INTO Members (MemberID, Age, Gender, MembershipType) VALUES (1, 35, 'Female', 'Premium'), (2, 45, 'Male', 'Basic'), (3, 28, 'Female', 'Premium'), (4, 32, 'Male', 'Premium'), (5, 48, 'Female', 'Basic'); CREATE TABLE ClassAttendance (MemberID INT, Class VARCHAR(20), Date DATE); INSERT INTO ClassAttendance (MemberID, Class, Date) VALUES (1, 'Cycling', '2022-04-01'), (2, 'Yoga', '2022-04-02'), (3, 'Cycling', '2022-04-03'), (4, 'Yoga', '2022-04-04'), (5, 'Pilates', '2022-04-05'), (1, 'Cycling', '2022-04-06'), (2, 'Yoga', '2022-04-07'), (3, 'Cycling', '2022-04-08'), (4, 'Yoga', '2022-04-09'), (5, 'Pilates', '2022-04-10');\n\nQuestion: How many members attended each type of class in Ap

In [14]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# ============================================
# Step 1: 配置 LoRA
# ============================================
peft_config = LoraConfig(
    r=8,                    # LoRA 的 rank,越大越能学复杂模式,但也更耗显存
    lora_alpha=16,          # LoRA 的缩放系数,通常设成 r 的 2 倍
    lora_dropout=0.05,      # dropout,防止过拟合
    bias="none",            # 不训练 bias 参数
    task_type="CAUSAL_LM",  # 任务类型:因果语言模型(像 GPT 那样从左到右生成)
    target_modules="all-linear",  # 给所有线性层都加 LoRA(简单粗暴但有效)
)


# ============================================
# Step 2: 配置训练参数
# ============================================
training_args = SFTConfig(
    output_dir="./gemma-sql-lora",          # 训练产物保存位置
    num_train_epochs=1,                      # 训练 1 轮(epoch)
    per_device_train_batch_size=2,           # 每个 GPU 上每次喂 2 条样本
    gradient_accumulation_steps=4,           # 累积 4 个 batch 再更新一次梯度(等效于 batch_size=8)
    gradient_checkpointing=True,             # 用时间换显存,显存吃紧时必开
    optim="paged_adamw_8bit",                # 8bit 优化器,省显存
    learning_rate=2e-4,                      # LoRA 微调常用学习率
    lr_scheduler_type="cosine",              # 学习率调度:余弦衰减
    warmup_ratio=0.1,                        # 前 10% 的步数学习率从 0 慢慢升上来
    logging_steps=20,                        # 每 20 步打印一次日志
    save_strategy="epoch",                   # 每个 epoch 结束保存
    eval_strategy="epoch",                   # 每个 epoch 结束评估一次
    bf16=True,                               # 用 bfloat16 训练,T4 不支持就换 fp16=True
    report_to="none",                        # 不上报到 wandb/tensorboard
    max_length=1024,                         # 单条样本最长 1024 个 token,超过截断
    packing=False,                           # 不把多条样本打包成一条
)


# ============================================
# Step 3: 创建 Trainer 并开始训练
# ============================================
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print("✅ Trainer 创建完成,准备开始训练")
print(f"训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(eval_dataset)}")
print(f"\n开始训练...")

# 启动训练!
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


✅ Trainer 创建完成,准备开始训练
训练集大小: 5000
验证集大小: 200

开始训练...


Epoch,Training Loss,Validation Loss
1,0.488026,0.448435


TrainOutput(global_step=625, training_loss=0.5948646461486816, metrics={'train_runtime': 4277.476, 'train_samples_per_second': 1.169, 'train_steps_per_second': 0.146, 'total_flos': 4861251486059520.0, 'train_loss': 0.5948646461486816})

In [15]:
# 用微调后的模型(model 当前已经是 trainer 训练完的状态)
# 重新测试之前那 4 条 baseline 样本

print("="*70)
print("🔥 微调后模型测试 vs Baseline")
print("="*70)

test_indices = [0, 100, 1000, 5000]

for idx in test_indices:
    test_sample = dataset[idx]  # 注意:用原始 dataset(没格式化的)

    user_prompt = f"""Database schema:
{test_sample['sql_context']}

Question: {test_sample['sql_prompt']}

Generate the SQL query:"""

    messages = [
        {"role": "user", "content": system_message + "\n\n" + user_prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    input_length = inputs['input_ids'].shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    print(f"\n{'='*70}")
    print(f"🔢 样本 {idx}  |  领域: {test_sample['domain']}  |  复杂度: {test_sample['sql_complexity']}")
    print(f"{'='*70}")
    print(f"📝 问题: {test_sample['sql_prompt']}")
    print(f"\n🎯 标准答案:\n{test_sample['sql']}")
    print(f"\n🤖 微调后 Gemma 回答:\n{response}")

🔥 微调后模型测试 vs Baseline

🔢 样本 0  |  领域: forestry  |  复杂度: single join
📝 问题: What is the total volume of timber sold by each salesperson, sorted by salesperson?

🎯 标准答案:
SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;

🤖 微调后 Gemma 回答:
SELECT s.name, SUM(ts.volume) as total_volume FROM timber_sales ts JOIN salesperson s ON ts.salesperson_id = s.salesperson_id GROUP BY s.name;

🔢 样本 100  |  领域: sustainable infrastructure  |  复杂度: subqueries
📝 问题: What is the maximum number of green buildings in each state in the US, constructed before 2015?

🎯 标准答案:
SELECT state, MAX(cnt) FROM (SELECT state, COUNT(*) AS cnt FROM green_buildings_us WHERE construction_year < 2015 GROUP BY state) AS subquery;

🤖 微调后 Gemma 回答:
SELECT state, MAX(construction_year) FROM green_buildings_us WHERE construction_year < 2015 GROUP BY state;

🔢 样本 1000  |  领域: 

In [16]:
# 保存 LoRA adapter 到本地
save_path = "./gemma-sql-lora-final"

# trainer.save_model 会保存:
# - adapter_config.json:LoRA 配置
# - adapter_model.safetensors:LoRA 权重(只有几十 MB,因为只是 adapter)
# - tokenizer 相关文件
trainer.save_model(save_path)

print(f"✅ 模型已保存到 {save_path}")

# 看看保存的文件大小
import os
total_size = 0
print(f"\n📦 保存的文件:")
for f in os.listdir(save_path):
    file_path = os.path.join(save_path, f)
    size_mb = os.path.getsize(file_path) / 1024 / 1024
    total_size += size_mb
    print(f"  {f}: {size_mb:.2f} MB")
print(f"\n总大小: {total_size:.2f} MB")

✅ 模型已保存到 ./gemma-sql-lora-final

📦 保存的文件:
  tokenizer.json: 31.84 MB
  adapter_model.safetensors: 12.49 MB
  README.md: 0.00 MB
  chat_template.jinja: 0.00 MB
  training_args.bin: 0.01 MB
  adapter_config.json: 0.00 MB
  tokenizer_config.json: 0.00 MB

总大小: 44.34 MB


In [17]:
# 把整个保存目录打包成 zip,方便下载
import shutil
shutil.make_archive("/content/gemma-sql-lora-final", "zip", "/content/gemma-sql-lora-final")

# 触发浏览器下载
from google.colab import files
files.download("/content/gemma-sql-lora-final.zip")

print("✅ 开始下载,文件会出现在你的'下载'文件夹")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 开始下载,文件会出现在你的'下载'文件夹
